# VNL 2024 Men — Data Analysis Notebook
> **Volleyball Nations League 2024 · Men's Tournament**  
> Data: 16 teams · 304 players · 7 statistical categories

---

## Sections
1. Setup & Data Loading
2. Team Performance
3. Best Attacker
4. Best Libero (Defence)
5. Best Middle Blocker
6. Best Server
7. Team Synergy & Balance
8. Player Demographics

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.data_loader    import build_master, team_summary
from src.rankings       import all_rankings, score_attackers, score_liberos, score_middle_blockers, score_servers
from src.analysis       import team_synergy, player_contribution, attack_risk_profile, age_analysis, height_analysis

%matplotlib inline
plt.rcParams['figure.dpi'] = 130
plt.rcParams['figure.facecolor'] = 'white'

master   = build_master()
team_df  = team_summary(master)
rankings = all_rankings(master)

print(f'Master: {master.shape}')
master.head(3)

## 1. Team Performance

In [ ]:
team_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total points
ax = axes[0]
sorted_t = team_df.sort_values('Total_Pts')
ax.barh(sorted_t['Team'], sorted_t['Total_Pts'], color='#2B6CB0')
ax.set_title('Total Points per Team', fontweight='bold')
ax.set_xlabel('Points')

# Breakdown stacked
ax = axes[1]
x = range(len(team_df))
ax.bar(x, team_df['Total_Atk'],   label='Attack',  color='#2B6CB0')
ax.bar(x, team_df['Total_Block'], bottom=team_df['Total_Atk'], label='Block', color='#276749')
ax.bar(x, team_df['Total_Serve'], bottom=team_df['Total_Atk']+team_df['Total_Block'], label='Serve', color='#C05621')
ax.set_xticks(list(x))
ax.set_xticklabels(team_df['Team'], rotation=45, ha='right', fontsize=9)
ax.legend()
ax.set_title('Points Breakdown', fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Best Attackers

In [ ]:
rankings['attackers'].head(10)

In [ ]:
atk = rankings['attackers']

fig, ax = plt.subplots(figsize=(9, 6))
pos_color = {'O':'#2B6CB0', 'OH':'#C05621', 'MB':'#553C9A'}
for pos, grp in atk.groupby('Position'):
    ax.scatter(grp['Pt_Attack'], grp['p_Attack'],
               c=pos_color.get(pos, '#4A5568'), label=pos, s=60, alpha=0.75)

for _, r in atk.head(10).iterrows():
    ax.annotate(r['Name'], (r['Pt_Attack'], r['p_Attack']),
                textcoords='offset points', xytext=(4,3), fontsize=7)

ax.set_xlabel('Attack Points (total)')
ax.set_ylabel('Efficiency %')
ax.set_title('Attack Volume vs Efficiency', fontweight='bold')
ax.legend(title='Position')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Best Libero (Defence)

In [ ]:
rankings['liberos']

In [ ]:
lib = rankings['liberos'].head(10)
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(lib['Sf_Dig'], lib['Sf_Receive'], s=80, color='#276749')
for _, r in lib.iterrows():
    ax.annotate(r['Name'], (r['Sf_Dig'], r['Sf_Receive']),
                textcoords='offset points', xytext=(4,3), fontsize=8)
ax.set_xlabel('Successful Digs')
ax.set_ylabel('Successful Receives')
ax.set_title('Top 10 Liberos — Digs vs Receives', fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Best Middle Blockers

In [ ]:
rankings['middle_blockers'].head(10)

## 5. Team Synergy & Balance

In [ ]:
syn = team_synergy(team_df)
syn

In [ ]:
# Stacked area-style phase share comparison
syn_sorted = syn.sort_values('balance_score', ascending=False).reset_index(drop=True)
x = range(len(syn_sorted))

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x, syn_sorted['atk_share']*100,   label='Attack %',  color='#2B6CB0')
ax.bar(x, syn_sorted['block_share']*100, bottom=syn_sorted['atk_share']*100,
       label='Block %', color='#276749')
ax.bar(x, syn_sorted['serve_share']*100,
       bottom=(syn_sorted['atk_share']+syn_sorted['block_share'])*100,
       label='Serve %', color='#C05621')
ax.axhline(100/3, color='black', linestyle='--', alpha=0.4, label='Equal share (33%)')
ax.set_xticks(list(x))
ax.set_xticklabels(syn_sorted['Team'], rotation=45, ha='right')
ax.set_ylabel('% of total points')
ax.set_title('Team Phase Composition (sorted by balance)', fontweight='bold')
ax.legend(loc='upper right', fontsize=8)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Player Demographics

In [ ]:
print('Average age by position:')
print(age_analysis(master)['avg_age_by_pos'])
print()
print('Height by position:')
print(height_analysis(master).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
master['Age'].hist(bins=15, ax=ax, color='#2B6CB0', edgecolor='white')
ax.set_title('Age Distribution', fontweight='bold')
ax.set_xlabel('Age'); ax.set_ylabel('Players')
ax.spines[['top','right']].set_visible(False)

ax = axes[1]
master['Height'].hist(bins=15, ax=ax, color='#276749', edgecolor='white')
ax.set_title('Height Distribution', fontweight='bold')
ax.set_xlabel('cm'); ax.set_ylabel('Players')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()